# 03 - Neural networks, nonlinear features, and overfitting

**Goal:** build a multilayer perceptron (MLP), explain why it needs an activation function,
and observe how training behavior differs from evaluation behavior.

**Before you start:** lesson 02's logits, cross-entropy, gradients, and train/validation/test
separation. This notebook creates its own data and model; previous outputs are not required.
Use **Restart Kernel and Run All Cells** for a clean demonstration.

**Live demonstration:** 45-60 minutes with discussion. Training time varies by hardware;
default subsets keep the experiment small. Smoke mode checks execution only.
We will inspect actual results rather than assume a deeper network always scores higher.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the project root or in notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the complete course folder; course_utils.py is required.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch
import course_utils as cu

DEVICE = cu.configure(seed=42)
SMOKE = cu.SMOKE  # COURSE_SMOKE=1: validation only; too small to judge model quality.
QUICK = True     # Set False for a longer learning experiment, then Restart Kernel and Run All.


## 1. Set a repeatable experiment

More parameters give a model more ways to fit the training set. They also make overfitting
possible. Keep the dataset, seed, and evaluation rule explicit so comparisons have meaning.


In [ ]:
from copy import deepcopy
from torch import nn
from torchvision import transforms

SEED = 42
cu.seed_everything(SEED)
TRAIN_N = 96 if SMOKE else (3000 if QUICK else 48000)
VAL_N = 48 if SMOKE else (800 if QUICK else 12000)
TEST_N = 48 if SMOKE else (1000 if QUICK else 10000)
EPOCHS = 1 if SMOKE else (6 if QUICK else 20)
BATCH_SIZE = 64
LR = 1e-3
WEIGHT_DECAY = 1e-4
print({"device": str(DEVICE), "epochs": EPOCHS, "smoke": SMOKE})


## 2. Prepare independent data splits

Each image starts as `[1, 28, 28]`. The fixed normalization changes `[0, 1]` to `[-1, 1]`.
Only the training loader shuffles. Validation and test loaders keep a stable order for
predictions and image galleries. The test set is reserved until the final assessment.


In [ ]:
preprocessing = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,)),
])
train_ds, val_ds, test_ds, class_names = cu.fashion_data(
    train_transform=preprocessing, eval_transform=preprocessing,
    train_limit=TRAIN_N, val_limit=VAL_N, test_limit=TEST_N, seed=SEED,
)
train_loader = cu.loader(train_ds, BATCH_SIZE, shuffle=True, seed=SEED)
val_loader = cu.loader(val_ds, BATCH_SIZE)
test_loader = cu.loader(test_ds, BATCH_SIZE)
images, labels = next(iter(train_loader))
print(images.shape, labels.shape)
cu.show_images(train_ds, class_names, n=8, mean=(0.5,), std=(0.5,))


## 3. Why stacking linear layers is not enough

If `h = x @ W1.T + b1` and `out = h @ W2.T + b2`, substitution gives another single
linear layer: `W = W2 @ W1`, `b = W2 @ b1 + b2`.
A second linear layer alone cannot create a nonlinear decision boundary.
Run this small numerical check before adding an activation.


In [ ]:
first = nn.Linear(2, 3)
second = nn.Linear(3, 1)
points = torch.randn(5, 2)
with torch.no_grad():
    stacked = second(first(points))
    combined_weight = second.weight @ first.weight
    combined_bias = second.weight @ first.bias + second.bias
    combined = points @ combined_weight.T + combined_bias
print("Largest rounding difference:", (stacked - combined).abs().max().item())
assert torch.allclose(stacked, combined, atol=1e-6)


## 4. Add a nonlinear activation: ReLU

ReLU means `max(0, x)`: negative values become zero and positive values pass through.
Putting ReLU between linear layers makes the network able to represent nonlinear functions.
A hidden unit's value is a learned feature; it is not necessarily a human-readable concept.

**Pause and predict:** what will ReLU return for `[-2, -1, 0, 1, 2]`?


In [ ]:
values = torch.linspace(-3, 3, 121)
plt.figure(figsize=(6, 3))
plt.plot(values.numpy(), torch.relu(values).numpy(), label="ReLU(x)")
plt.axhline(0, color="gray", linewidth=0.7)
plt.axvline(0, color="gray", linewidth=0.7)
plt.xlabel("Input")
plt.ylabel("Output")
plt.legend()
plt.grid(alpha=0.2)
plt.show()
print(torch.relu(torch.tensor([-2., -1., 0., 1., 2.])))


## 5. Define an MLP with explicit dimensions

The path is `[N,1,28,28] -> [N,784] -> [N,128] -> [N,64] -> [N,10]`.
Dropout randomly zeros some hidden activations **during training**. At evaluation time it
becomes an identity operation. Its purpose is regularization, not compression.

The final layer returns logits. There is no ReLU or softmax after it because cross-entropy
expects unrestricted raw scores. `nn.Module` registers the layers so their parameters are trained.


In [ ]:
class FashionMLP(nn.Module):
    def __init__(self, hidden=128, dropout=0.2):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.network(x)

cu.seed_everything(SEED)
model = FashionMLP().to(DEVICE)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
with torch.no_grad():
    print("Output shape:", model(images.to(DEVICE)).shape)


## 6. Observe dropout directly

PyTorch scales the retained activations during training so their expected value is unchanged.
Two training passes can produce different outputs even when the input and weights are unchanged.
`eval()` controls dropout; `no_grad()` controls gradient recording. They solve different problems.


In [ ]:
dropout_demo = nn.Dropout(p=0.5)
ones = torch.ones(12)
dropout_demo.train()
print("Training pass A:", dropout_demo(ones).tolist())
print("Training pass B:", dropout_demo(ones).tolist())
dropout_demo.eval()
print("Evaluation pass:", dropout_demo(ones).tolist())

model.eval()
with torch.no_grad():
    output_a = model(images.to(DEVICE))
    output_b = model(images.to(DEVICE))
print("Evaluation outputs match:", torch.allclose(output_a, output_b))


## 7. Write reusable, readable train and validation functions

This is the same learning cycle from lesson 02, organized into functions.
AdamW adapts updates for different parameters and applies weight decay. Weight decay and
dropout are separate regularization choices; neither guarantees better validation performance.

Read `train_one_epoch` from top to bottom. Identify where gradients are cleared, calculated,
and used. Then find why `validate` cannot update the network.


In [ ]:
criterion = nn.CrossEntropyLoss()

def train_one_epoch(network, data_loader, optimizer):
    network.train()
    loss_sum, correct, count = 0.0, 0, 0
    for x, y in data_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = network(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        count += y.size(0)
    return {"loss": loss_sum / count, "accuracy": correct / count}

@torch.no_grad()
def validate(network, data_loader):
    network.eval()
    loss_sum, correct, count = 0.0, 0, 0
    for x, y in data_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = network(x)
        loss_sum += criterion(logits, y).item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        count += y.size(0)
    return {"loss": loss_sum / count, "accuracy": correct / count}

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
print("Initial validation:", validate(model, val_loader))


## 8. Train and restore the best validation checkpoint

Choose the checkpoint with the lowest validation loss. The last epoch is not automatically
the best. Copying `state_dict()` preserves the weights at that moment.
We run a fixed number of epochs for a predictable demonstration; early stopping is a later extension.


In [ ]:
history = []
best_val_loss = float("inf")
best_state = deepcopy(model.state_dict())
for epoch in range(1, EPOCHS + 1):
    train_metrics = train_one_epoch(model, train_loader, optimizer)
    val_metrics = validate(model, val_loader)
    row = {"epoch": epoch, "train_loss": train_metrics["loss"],
           "train_accuracy": train_metrics["accuracy"],
           "val_loss": val_metrics["loss"], "val_accuracy": val_metrics["accuracy"]}
    history.append(row)
    if val_metrics["loss"] < best_val_loss:
        best_val_loss = val_metrics["loss"]
        best_state = deepcopy(model.state_dict())
    print(f"Epoch {epoch:02d}: train={train_metrics['loss']:.3f}, "
          f"val={val_metrics['loss']:.3f}, val accuracy={val_metrics['accuracy']:.1%}")
model.load_state_dict(best_state)
print("Best checkpoint restored:", validate(model, val_loader))


## 9. Diagnose fitting behavior

If both training and validation performance stay poor, possible causes include too few
updates, a poor learning rate, an input bug, or insufficient model capacity. If training
improves while validation deteriorates, investigate overfitting and distribution differences.

Training metrics include active dropout and changing weights. Validation metrics use dropout
disabled and fixed weights. Validation accuracy can therefore exceed training accuracy,
especially early on; that observation alone is not evidence of data leakage.


In [ ]:
cu.plot_history(history)
# Compare both splits using the same fixed model with dropout disabled.
train_eval_loader = cu.loader(train_ds, BATCH_SIZE, shuffle=False)
train_eval = validate(model, train_eval_loader)
val_eval = validate(model, val_loader)
print("Training set, evaluation mode:", train_eval)
print("Validation set, evaluation mode:", val_eval)
print(f"Accuracy gap: {train_eval['accuracy'] - val_eval['accuracy']:+.1%}")


## 10. A useful debugging experiment: fit one tiny batch

A small network should usually be able to fit a tiny collection of consistent examples
after enough updates. If it cannot, inspect shapes, labels, preprocessing, learning rate,
gradients, and whether the optimizer contains the model's actual parameters.

This is a **separate throwaway model**, trained repeatedly on the same training batch with
dropout off. It demonstrates memorization, not generalization. Smoke mode uses too few
steps to draw conclusions; even the quick run need not reach perfect accuracy.


In [ ]:
cu.seed_everything(SEED)
tiny_model = FashionMLP(hidden=64, dropout=0.0).to(DEVICE)
tiny_optimizer = torch.optim.Adam(tiny_model.parameters(), lr=0.01)
tiny_x, tiny_y = images[:16].to(DEVICE), labels[:16].to(DEVICE)
tiny_steps = 3 if SMOKE else (80 if QUICK else 200)
tiny_losses = []
tiny_model.train()
for step in range(tiny_steps):
    tiny_optimizer.zero_grad(set_to_none=True)
    tiny_loss = criterion(tiny_model(tiny_x), tiny_y)
    tiny_loss.backward()
    tiny_optimizer.step()
    tiny_losses.append(tiny_loss.item())
tiny_model.eval()
with torch.no_grad():
    tiny_accuracy = (tiny_model(tiny_x).argmax(1) == tiny_y).float().mean().item()
plt.figure(figsize=(6, 3))
plt.plot(tiny_losses)
plt.xlabel("Repeated updates on the same 16 examples")
plt.ylabel("Cross-entropy")
plt.show()
print(f"Tiny training batch accuracy: {tiny_accuracy:.1%}")
del tiny_model, tiny_optimizer


## 11. Assess the main model once on held-out test images

The tiny-batch demonstration did not modify the main model. Now use its selected validation
checkpoint for the final test estimate. Inspect a prediction as well as the aggregate score.
Softmax scores are useful for display, but confidence calibration needs separate evaluation.


In [ ]:
test_logits, test_labels = cu.predict(model, test_loader, DEVICE)
test_predictions = test_logits.argmax(1)
test_accuracy = (test_predictions == test_labels).float().mean().item()
print(f"Final test accuracy: {test_accuracy:.1%} ({len(test_labels)} examples)")
print(f"Final test loss: {criterion(test_logits, test_labels).item():.4f}")
first_image, actual = test_ds[0]
top_scores, top_ids = test_logits[0].softmax(0).topk(3)
plt.figure(figsize=(3, 3))
plt.imshow((first_image[0] * 0.5 + 0.5).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
plt.title(f"Actual: {class_names[actual]}")
plt.axis("off")
plt.show()
for score, class_id in zip(top_scores.tolist(), top_ids.tolist()):
    print(f"{class_names[class_id]:15s} softmax score={score:.3f}")


## 12. Save the selected weights with their context

A weight file alone does not describe preprocessing or the order of class names.
Save that context with the architecture settings. This notebook writes a new named artifact
that later lessons do not require; rerunning replaces this lesson's artifact.
Only load checkpoint files you trust.


In [ ]:
checkpoint_path = cu.ARTIFACTS / "03_fashion_mlp.pt"
torch.save({
    "model_state": {key: value.detach().cpu() for key, value in model.state_dict().items()},
    "architecture": "FashionMLP", "hidden": 128, "dropout": 0.2,
    "class_names": list(class_names), "image_size": [28, 28],
    "mean": [0.5], "std": [0.5], "seed": SEED,
    "smoke": SMOKE, "train_examples": len(train_ds),
}, checkpoint_path)
print("Saved:", checkpoint_path)


## 13. Exercises and discussion

1. Predict the parameter count of the first `Linear(784, 128)` layer, then verify it.
2. Run a controlled comparison with dropout values `0.0` and `0.3`. Use the same initial
   seed, data split, epoch budget, and selection rule; judge with validation data.
3. Replace both ReLUs with `nn.Tanh()`. Explain why this remains a nonlinear model.
4. Explain why excellent accuracy on the 16-image experiment does not establish usefulness.

<details><summary>Solution hints</summary>

The first layer contains `784 * 128 + 128 = 100,480` trainable numbers.
Recreate the model, optimizer, and shuffled loader for each dropout condition; do not keep
training the previous model and call the result a fair comparison. Several seeds provide
stronger evidence than one run. `Tanh` is nonlinear, though its gradients and scale differ.
The tiny model has seen the exact evaluation examples repeatedly.

```python
layer = nn.Linear(784, 128)
print(sum(parameter.numel() for parameter in layer.parameters()))
# For a fresh controlled run:
cu.seed_everything(SEED)
candidate = FashionMLP(hidden=128, dropout=0.3).to(DEVICE)
candidate_loader = cu.loader(train_ds, BATCH_SIZE, shuffle=True, seed=SEED)
candidate_optimizer = torch.optim.AdamW(candidate.parameters(), lr=LR,
                                        weight_decay=WEIGHT_DECAY)
# Repeat steps 8-9 with candidate, candidate_loader, and candidate_optimizer.
```

</details>

**Presenter checkpoint:** ask a learner to explain the different purposes of `eval()` and
`no_grad()`, and why raw logits go into cross-entropy.

**Limitations:** flattening loses an explicit representation of spatial neighborhoods.
We have not searched architectures systematically, and a quick subset is not a benchmark.
Next, use convolutional layers to share filters across image locations.

**References:** [PyTorch model tutorial](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html),
[Dropout](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html),
[AdamW](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html).
